# Extended Hubbard model QPE with `qdk-chemistry`

This notebook demonstrates a complete workflow from deriving an [Extended Hubbard model](https://en.wikipedia.org/wiki/Hubbard_model) Hamiltonian for cyclobutadiene's π-system to estimating its ground state energy using iterative Quantum Phase Estimation (iQPE). 
Cyclobutadiene in its square ($D_{4h}$) geometry is a transition state connecting two rectangular Jahn-Teller-distorted minima, making it an interesting strongly correlated test case. The Extended Hubbard model augments the Hubbard model with nearest-neighbour intersite Coulomb repulsion, making it well-suited for conjugated π-systems where both on-site and short-range electron-electron interactions are important. This is one example of a wide range of functionality in `qdk-chemistry`. Please see <https://github.com/microsoft/qdk-chemistry> for the full documentation.

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md), you will need to install the `jupyter` extra to run this notebook:

```bash
pip install 'qdk-chemistry[jupyter]'
```

This installs the additional dependencies required by this notebook (ipykernel, pandas, and pyscf).

In [ ]:
# Load frequently used external packages
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

# Reduce logging output for demo
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)

## From molecular structure to Extended Hubbard parameters

The Extended Hubbard model adds electron-electron interactions to the tight-binding (Hückel) Hamiltonian. In addition to the hopping term $t$, it includes on-site Coulomb repulsion $U$ and nearest-neighbour intersite repulsion $V$:

$$\hat{H}_\text{Ext. Hubbard} = \varepsilon \sum_i \hat{n}_i - t\sum_{\langle i,j \rangle} (\hat{a}_i^\dagger \hat{a}_j + \text{h.c.}) + U \sum_i \hat{n}_{i\uparrow} \hat{n}_{i\downarrow} + \frac{1}{2}\sum_{\langle i,j \rangle} V_{ij}\, (\hat{n}_i - z_i)(\hat{n}_j - z_j)$$

Here $\hat{n}_i = \hat{n}_{i\uparrow} + \hat{n}_{i\downarrow}$ is the total number operator on site $i$, and $z_i$ is the effective core charge (typically $z_i = 1$ for one π-electron per carbon). The term $(\hat{n}_i - z_i)$ measures the net charge deviation at each site. For a comprehensive review of the Extended Hubbard model and its parent PPP Hamiltonian, see [Fabian, Glaser and Solomon, *Digital Discovery*, 2026, **5**, 482–496](https://doi.org/10.1039/D5DD00445D).

Rather than choosing parameters by hand, we derive $t$ and $U$ from the square cyclobutadiene molecule using `qdk-chemistry`, and compute the nearest-neighbour $V$ using the Ohno potential. The on-site orbital energy $\varepsilon$ is set to zero, as it only shifts the total energy by a constant and does not affect the physics.

The molecular structure is loaded from an [XYZ-format](https://en.wikipedia.org/wiki/XYZ_file_format) file. The 4 carbons in the square geometry are equidistant, which results in orbital degeneracies that cannot be resolved by mean-field approaches and requires a multi-determinantal description.

In [ ]:
from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import create
from qdk.widgets import MoleculeViewer

# Load cyclobutadiene structure from XYZ file (D4h square transition state)
structure = Structure.from_xyz_file(Path("data/cyclobutadiene.structure.xyz"))

# Visualize the cyclobutadiene structure
MoleculeViewer(molecule_data=structure.to_xyz())

## Identifying the π-orbitals

To parameterize the Extended Hubbard model, we need to identify the 4 carbon π-orbitals out of cyclobutadiene's full set of molecular orbitals. This is a common task in quantum chemistry and `qdk-chemistry` provides automated tools to accomplish it.

First we run a Hartree-Fock (HF) self-consistent field calculation to obtain a set of molecular orbitals, which will serve as input for the orbital selection step. For simplicity, we use the STO-3G basis set.

In [ ]:
# Run SCF
scf_solver = create("scf_solver")
E_hf, wfn_hf = scf_solver.run(structure, charge=0, spin_multiplicity=1, basis_or_guess="sto-3g")
print(f"Hartree-Fock energy: {E_hf:.6f} Hartree")

The π-system is built up by the $p_z$ orbitals of the carbons. Hence, we use PySCF's Automated Valence Active Space (AVAS) method to identify and select the relevant orbitals.

In [ ]:
import qdk_chemistry.plugins.pyscf  # Enable PySCF plugin  # noqa: F401

# Use AVAS to select the π-orbitals by projecting onto the carbon 2pz atomic orbitals
avas_selector = create("active_space_selector", "pyscf_avas", ao_labels=["C 2pz"])
avas_wfn = avas_selector.run(wfn_hf)
indices, _ = avas_wfn.get_orbitals().get_active_space_indices()
n_active_e = sum(avas_wfn.get_active_num_electrons())
print(f"AVAS selected CAS({n_active_e},{len(indices)}) active space")
print(f"  Active orbital indices: {list(indices)}")

We can verify the selected orbitals by visual inspection. Based on MO theory, we expect one orbital without any node, two degenerate orbitals with one node each and one orbital with two nodes for the delocalized canonical orbitals. 

In [ ]:
from qdk_chemistry.utils.cubegen import generate_cubefiles_from_orbitals

# Visualize the AVAS-selected active space orbitals (canonical)
active_orbitals = avas_wfn.get_orbitals()
cube_data = generate_cubefiles_from_orbitals(
    orbitals=active_orbitals,
    grid_size=(40, 40, 40),
    margin=10.0,
    indices=indices,
)

MoleculeViewer(molecule_data=structure.to_xyz(), cube_data=cube_data)

The canonical orbitals are delocalized over the ring. For the Extended Hubbard model we need site-localized orbitals, so we apply a Pipek-Mezey localization to produce one orbital per carbon atom, which will later represent a site in the model Hamiltonian. We can verify that the orbitals are localized on each carbon atom by visualization. 

In [ ]:
from qdk_chemistry.utils.cubegen import generate_cubefiles_from_orbitals

# Localize the active-space orbitals for clearer visualization
localizer = create("orbital_localizer", "qdk_pipek_mezey")
loc_wfn = localizer.run(avas_wfn, *avas_wfn.get_orbitals().get_active_space_indices())

# Visualize the localized active space orbitals
loc_orbitals = loc_wfn.get_orbitals()
loc_indices, _ = loc_orbitals.get_active_space_indices()
cube_data = generate_cubefiles_from_orbitals(
    orbitals=loc_orbitals,
    grid_size=(40, 40, 40),
    margin=10.0,
    indices=loc_indices,
)

MoleculeViewer(molecule_data=structure.to_xyz(), cube_data=cube_data)

## Extracting Extended Hubbard parameters from the active space

With the localized π-orbitals in hand, we construct the active-space Hamiltonian and extract the Extended Hubbard parameters:
- **Hopping integral** $t$: for each orbital, the largest absolute off-diagonal one-body element identifies its nearest neighbour(s) in the ring
- **On-site repulsion** $U$: average of the diagonal two-body integrals $\langle ii|ii \rangle$
- **Nearest-neighbour repulsion** $V$: computed from the Ohno potential using the ab initio $U$ and the C–C bond length

Note that the Pipek-Mezey localization does not preserve any particular ordering of the orbitals, so we cannot simply read off nearest-neighbour hoppings from adjacent matrix indices. Instead, we rely on the magnitude of the off-diagonal elements to identify the connections.

In [ ]:
n_sites = len(indices)
n_active_alpha, n_active_beta = avas_wfn.get_active_num_electrons()

# Construct the active-space Hamiltonian from the localized orbitals
hamiltonian_constructor = create("hamiltonian_constructor")
refined_orbitals = loc_wfn.get_orbitals()
active_hamiltonian = hamiltonian_constructor.run(refined_orbitals)

# Get one-body integrals of the active space
h1_alpha, _ = active_hamiltonian.get_one_body_integrals()
print(f"One-body integrals ({n_sites}x{n_sites}):\n{np.round(h1_alpha, 4)}")

# Hopping integral: identify nearest-neighbour hoppings from the largest
# off-diagonal elements in the one-body integral matrix.
nn_hoppings = []
for i in range(n_sites):
    row = np.abs(h1_alpha[i].copy())
    row[i] = 0.0  # exclude diagonal
    # The largest off-diagonal element is the nearest-neighbour hopping
    nn_hoppings.append(np.max(row))
t = float(np.mean(nn_hoppings))

# On-site Coulomb repulsion U: average of diagonal two-body integrals (i,i,i,i)
U_values = []
for i in range(n_sites):
    U_values.append(active_hamiltonian.get_two_body_element(i, i, i, i))
U = float(np.mean(U_values))

print(f"\nExtended Hubbard parameters from CAS({n_active_alpha + n_active_beta},{n_sites}) active space:")
print(f"  Hopping integral t = {t:.4f} Hartree")
print(f"  On-site repulsion U = {U:.4f} Hartree")

Cyclobutadiene's π-system has a ring topology, which we represent as a periodic 4-site chain. The nearest-neighbour intersite repulsion $V$ is computed using the Ohno potential with the ab initio $U$ and the C–C bond length. The Extended Hubbard Hamiltonian is then constructed using `create_ppp_hamiltonian` with `nearest_neighbor_only=True`.

In [ ]:
from qdk_chemistry.data import LatticeGraph
from qdk_chemistry.utils.model_hamiltonians import create_ppp_hamiltonian, ohno_potential

# Build periodic chain lattice (ring topology)
lattice = LatticeGraph.chain(n=n_sites, periodic=True)

# Compute nearest-neighbour C-C bond length from the structure (coordinates are in Bohr)
coords = structure.get_coordinates()
carbon_coords = coords[:n_sites]  # first n_sites atoms are carbons
cc_distances = [np.linalg.norm(carbon_coords[i] - carbon_coords[(i + 1) % n_sites]) for i in range(n_sites)]
R_CC_bohr = float(np.mean(cc_distances))
print(f"Average C-C bond length: {R_CC_bohr:.4f} Bohr")

# Compute nearest-neighbour intersite Coulomb repulsion V via Ohno potential
V = ohno_potential(lattice, U=U, R=R_CC_bohr, nearest_neighbor_only=True)

# Build Extended Hubbard Hamiltonian (PPP with nearest-neighbour V only)
# epsilon=0 because it only shifts all energies by a constant (see discussion above)
hamiltonian = create_ppp_hamiltonian(lattice, epsilon=0.0, t=t, U=U, V=V, z=1.0)

# Extract integrals for display
h1 = hamiltonian.get_one_body_integrals()[0]  # one-body integral matrix
Vij = V[0, 1]

print(f"Extended Hubbard model: {lattice.num_sites} sites (ring), {n_active_alpha}α + {n_active_beta}β electrons")
print(f"One-body integrals:\n{np.round(h1, 4)}")
print(f"Two-body integrals (U): {U:.4f}")
print(f"Two-body integrals (V): {Vij:.4f}")
print(f"Ratio U/t = {U / t:.2f}, V/t = {Vij / t:.2f}")

## Classical reference energy

We solve the Extended Hubbard model exactly using the CASCI multi-configuration calculator to obtain a reference energy for benchmarking QPE. The configuration weights reveal the structure of the ground state.

In [ ]:
from qdk.widgets import Histogram

# Exact diagonalization (CASCI)
mc_calculator = create("multi_configuration_calculator", "macis_cas")
e_exact, wfn_exact = mc_calculator.run(hamiltonian, n_active_alpha, n_active_beta)
print(f"Exact ground state energy: {e_exact:.6f} Hartree")

# Plot top configuration weights from the CASCI wavefunction
num_configurations = len(wfn_exact.get_active_determinants())
print(f"Total configurations in the CASCI wavefunction: {num_configurations}")
top_dets = wfn_exact.get_top_determinants()
display(
    Histogram(
        bar_values={k.to_string(): np.abs(v)**2 for k, v in top_dets.items()},
        items="top-25",
        sort="high-to-low",
    )
)

## Optimizing trial wavefunction loading onto a quantum computer

The multi-configuration wavefunction can serve as the trial state for the iQPE algorithm. However, this information needs to be loaded as a state on a quantum computer.

The amount of data loaded onto the quantum computer can be optimized by exploiting the sparsity of the wavefunction. We take the top determinants from the CASCI solution and check their overlap with the full wavefunction.

In [ ]:
from qdk_chemistry.data import Wavefunction, CasWavefunctionContainer

# For square cyclobutadiene, the ground state is dominated by two degenerate determinants (udud and dudu)
NUM_TRIAL_DETS = 2

# Select top determinants and renormalize
dets = wfn_exact.get_top_determinants(NUM_TRIAL_DETS)
total_weight = sum(np.abs(c)**2 for c in dets.values())
dets = {det: c / np.sqrt(total_weight) for det, c in dets.items()}

# Build trial wavefunction
det_keys = list(dets.keys())
coeffs = np.array(list(dets.values()))
wfn_trial = Wavefunction(CasWavefunctionContainer(coeffs, det_keys, wfn_exact.get_orbitals()))

# Compute fidelity with exact wavefunction
exact_coeffs = np.array([wfn_exact.get_coefficient(det) for det in det_keys])
fidelity = np.abs(np.vdot(exact_coeffs, coeffs))**2
print(f"Overlap (fidelity) of trial state with CASCI wavefunction: {fidelity:.4f}")

# Plot trial state configuration weights
display(
    Histogram(
        bar_values={k.to_string(): np.abs(v)**2 for k, v in dets.items()},
        sort="high-to-low",
    )
)

There are many ways to "load" this state onto a quantum computer. `qdk-chemistry` provides optimized state preparation methods that exploit the structure of chemistry wavefunctions to reduce the number of operations and improve noise resilience.

In [ ]:
from qdk.widgets import Circuit

# Generate state preparation circuit for the sparse state via GF2+X sparse isometry
state_prep = create("state_prep", "sparse_isometry_gf2x")
state_prep_circuit = state_prep.run(wfn_trial)

# Visualize the sparse isometry circuit
display(Circuit(state_prep_circuit.get_qsharp_circuit()))

## Estimating the ground state energy with iterative quantum phase estimation

Kitaev-style iterative quantum phase estimation (iQPE) estimates an eigenphase of the time-evolution operator $\hat{U} = e^{-i\hat{H}t}$ using one ancilla qubit and a sequence of controlled-$\hat{U}^{2^k}$ applications.

Each iteration measures one bit of the phase (from most-significant to least-significant) and uses phase feedback to refine the estimate.

The Extended Hubbard Hamiltonian must be mapped to a qubit Hamiltonian that can be measured on a quantum computer.
The Jordan-Wigner transformation is a popular mapping that is used in this example.

In [ ]:
# Prepare the qubit-mapped Hamiltonian
qubit_mapper = create("qubit_mapper", algorithm_name="qdk", encoding="jordan-wigner")
qubit_hamiltonian = qubit_mapper.run(hamiltonian)
print("Qubit Hamiltonian:\n", qubit_hamiltonian.get_summary())

In [ ]:
# Set up parameters for iQPE
from utils.qpe_utils import compute_evolution_time

M_PRECISION = 10
SHOTS_PER_BIT = 3
SIMULATOR_SEED = 42

# For model Hamiltonians (no core energy), use the base time pi/||H|| without refinement
evolution_time = compute_evolution_time(qubit_hamiltonian, num_bits=M_PRECISION)
print(f"Proposed evolution time: {evolution_time:.4f} Hartree^-1")

The circuit for iQPE consists of initial trial state preparation followed by multiple controlled time-evolution operations. The time evolution is approximated using a 2nd-order Trotter-Suzuki decomposition, which provides better accuracy per Trotter step compared to the simpler 1st-order product formula. This cell visualizes one iteration of the iQPE circuit using built-in `qdk` visualization tools.

In [ ]:
# Use factory methods to create the iQPE algorithm components
evolution_builder = create("time_evolution_builder", "trotter", order=2)
circuit_mapper = create("controlled_evolution_circuit_mapper", "pauli_sequence")
iqpe = create(
    "phase_estimation",
    "iterative",
    num_bits=M_PRECISION,
    evolution_time=evolution_time,
    shots_per_bit=SHOTS_PER_BIT,
)

# Generate the iQPE iteration circuit for a specific iteration (3rd from last)
iqpe_iter_circuit = iqpe.create_iteration_circuit(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
    iteration=M_PRECISION - 3,
    total_iterations=M_PRECISION,
)

# Visualize the iQPE iteration circuit
display(Circuit(iqpe_iter_circuit.get_qsharp_circuit()))

This real-time example performs a single-trial iQPE run on the `qdk` full state simulator.

In [ ]:
# Execute the iQPE algorithm for a single trial
circuit_executor = create("circuit_executor", "qdk_full_state_simulator", seed=SIMULATOR_SEED)
result = iqpe.run(
    state_preparation=state_prep_circuit,
    qubit_hamiltonian=qubit_hamiltonian,
    circuit_executor=circuit_executor,
    evolution_builder=evolution_builder,
    circuit_mapper=circuit_mapper,
)

# Summarize the QPE results
estimated_energy = result.raw_energy + hamiltonian.get_core_energy()
estimated_error = abs(estimated_energy - e_exact)
print(f"QPE Results for {M_PRECISION}-bit precision:")
print(f"Reference CASCI energy: {e_exact:.6f} Hartree")
print(f"Total energy from phase estimation: {estimated_energy:.6f} Hartree")
print(f"Energy difference with CASCI energy: {estimated_error:.3e} Hartree")

The above cell used a single trial for a real-time example. However, iQPE generally requires multiple trials to establish confidence in the resulting estimate. The following cell performs a multi-trial iQPE run with high precision using the same simulator.

In [ ]:
NUM_TRIALS = 20
RESULTS_DIR = Path(
    f"results_iqpe_cyclobutadiene/precision_{M_PRECISION}/time_{round(evolution_time, 12)}"
)

# Run iQPE if results do not already exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for trial in range(NUM_TRIALS):
    trial_seed = SIMULATOR_SEED + trial
    json_path = RESULTS_DIR / f"iqpe_result_{trial_seed}.qpe_result.json"
    if not json_path.exists():
        print(f"Running trial {trial + 1} of {NUM_TRIALS}...")
        executor = create(
            "circuit_executor",
            "qdk_full_state_simulator",
            type="cpu",
            seed=trial_seed,
        )
        result = iqpe.run(
            state_preparation=state_prep_circuit,
            qubit_hamiltonian=qubit_hamiltonian,
            circuit_executor=executor,
            evolution_builder=evolution_builder,
            circuit_mapper=circuit_mapper,
        )
        result.to_json_file(json_path)

For a system with noise or an imperfect trial state, multiple trials of iQPE are needed to obtain a reliable estimate of the ground state energy. This estimate is typically taken as the most frequently observed energy from multiple trials ("majority vote").

In [ ]:
from qdk_chemistry.data import QpeResult

# Load the results
results_loaded = []
for json_file in RESULTS_DIR.glob("*qpe_result.json"):
    result = QpeResult.from_json_file(json_file)
    results_loaded.append(result)

# Count the energy values
energy_counts = Counter(
    [
        result.raw_energy + hamiltonian.get_core_energy()
        for result in results_loaded
    ]
)
print(f"QPE Results of {M_PRECISION} bit precision from {NUM_TRIALS} trials:")
display(pd.DataFrame(energy_counts.items(), columns=['Energy (Hartree)', 'Counts']))

# Use the most frequently observed energy across all trials as the overall estimate
estimated_energy, _ = energy_counts.most_common(1)[0]

The iQPE energy estimate accuracy is useful for benchmarking the impact of precision, evolution time, and other parameters on the final result. The following cell summarizes energy errors from the multiple trials.

In [ ]:
# Print summary of results
print(f"Reference CASCI energy: {e_exact:.6f} Hartree")
print(f"Total energy from phase estimation: {estimated_energy:.6f} Hartree")
print(f"Energy difference with CASCI energy: {abs(estimated_energy - e_exact):.3e} Hartree")

# Summarize the energy errors
energy_errors = {
    qpe_e - e_exact: count
    for qpe_e, count in sorted(energy_counts.items())
}

# Plot distribution of energy differences
print("Energy difference (Hartree) distribution:")
display(
    Histogram(
        bar_values={f"{err:.3e}": count for err, count in energy_errors.items()}
    )
)